# ETL MV Agent-Only Flow

本 notebook 用于 Agent-only 原型联调：依次运行 `SQLLoaderAgent`、`FeatureAgent`、`FamilyAgent`、`BatchClusterAgent`、`RewriteAgent`、`BatchMVAgent`、`ExecutorAgent` 和 `SelfIterationAgent`。

`ExecutorAgent` 当前只做 dry-run：不真实运行 SQL，只维护可用 MV 状态并输出当前 batch 的运行顺序 JSON。

In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

# 定位项目根目录，保证从 notebook 子目录运行时也能导入 llm_demo 包。
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'MV_gen' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

# 将项目根目录加入 Python import path，避免依赖 notebook 的启动目录。
sys.path.insert(0, str(PROJECT_ROOT))

# 只从项目根目录 .env 读取 DeepSeek 配置，不在 notebook 中硬编码 key。
load_dotenv(PROJECT_ROOT / '.env')

In [ ]:
from datetime import datetime
from llm_demo.src.core.artifact_store import ArtifactStore
from llm_demo.src.core.llm_client import LLMClient
from llm_demo.src.agents.sql_loader_agent import SQLLoaderAgent
from llm_demo.src.agents.feature_agent import FeatureAgent
from llm_demo.src.agents.family_agent import FamilyAgent
from llm_demo.src.agents.batch_cluster_agent import BatchClusterAgent
from llm_demo.src.agents.rewrite_agent import RewriteAgent
from llm_demo.src.agents.batch_mv_agent import BatchMVAgent
from llm_demo.src.agents.executor_agent import ExecutorAgent
from llm_demo.src.agents.self_iteration_agent import SelfIterationAgent

# 每次实验使用一个独立 run_id，所有 Artifact 都写入 llm_demo/artifacts/{run_id}/。
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
store = ArtifactStore(run_id=run_id, artifact_root=PROJECT_ROOT / 'llm_demo' / 'artifacts')

# LLMClient 会从 .env 读取 DeepSeek API 配置，并提供 JSON 输出解析能力。
llm_client = LLMClient(project_root=PROJECT_ROOT)

In [ ]:
# 使用 q42/q52 验证 QueryBlock 提取、QueryFamily 聚合、batch clustering、rewrite、MV candidate 和 dry-run 执行顺序。
sql_paths = [
    PROJECT_ROOT / 'tpcds-spark' / 'q42.sql',
    PROJECT_ROOT / 'tpcds-spark' / 'q52.sql',
]

# 1. 读取并校验原始 SQL 路径，只在 00_raw_sql/sql_manifest.json 中记录路径和摘要，不复制 SQL 文件。
sql_manifest_path = SQLLoaderAgent(store).run(sql_paths)

# 2. 调用 LLM + rules，从 manifest 指向的 SQL Text 提取 QueryBlock、query_to_qbs 和 qb_to_query。
query_blocks_path = FeatureAgent(store, llm_client).run(sql_manifest_path)

# 3. 基于 QueryBlock 的 family_key / join skeleton 聚合 QueryFamily。
families_path = FamilyAgent(store, llm_client).run(query_blocks_path)

# 4. 根据 QueryBlock 和 QueryFamily 生成 SQL 级 global batch。
batches_path = BatchClusterAgent(store, llm_client).run(query_blocks_path, families_path)

# 5. historical rewrite：在当前 batch 物化 MV 之前，先用历史可用 MV 尝试改写；没有可用 MV 时会生成 fallback rewritten SQL。
historical_rewrite_dir = RewriteAgent(store, llm_client).run(
    3,
    'historical',
    batches_path,
    sql_manifest_path,
    query_blocks_path,
)

# 6. BatchMVAgent 只在当前 batch 内生成 MV Candidate；第一阶段草稿不落盘，最终保存 evaluate 后结果。
mv_candidates_path = BatchMVAgent(store, llm_client).run(
    3,
    batches_path,
    query_blocks_path,
    families_path,
    historical_rewrite_dir,
)

# 7. ExecutorAgent dry-run 物化 MV：不执行 SQL，但把可物化 Candidate 写入 materialized_mvs.json。
materialized_mvs_path = ExecutorAgent(store).materialize_mvs(
    3,
    mv_candidates_path,
    store.path('04_batch_mvs/batch_3_mv_build.sql'),
)

# 8. final rewrite：使用当前 batch dry-run 物化后的完整 MV 状态重新改写 original SQL。
final_rewrite_dir = RewriteAgent(store, llm_client).run(
    3,
    'final',
    batches_path,
    sql_manifest_path,
    query_blocks_path,
    materialized_mvs_path,
)

# 9. ExecutorAgent dry-run 查询执行：只输出 final rewritten SQL 的运行顺序 JSON。
execution_order_path = ExecutorAgent(store).run_queries(3, final_rewrite_dir, batches_path)

# 10. 读取 run log，生成带证据引用的自迭代反馈建议。
feedback_path = SelfIterationAgent(store, llm_client).run(store.run_log_path)

In [ ]:
# 查看本次 run 的核心 Artifact 路径，便于人工检查 LLM 输出是否合理。
print('run_id:', run_id)
print('query_blocks:', query_blocks_path)
print('families:', families_path)
print('batches:', batches_path)
print('mv_candidates:', mv_candidates_path)
print('materialized_mvs:', materialized_mvs_path)
print('final_rewrite_dir:', final_rewrite_dir)
print('execution_order:', execution_order_path)
print('feedback:', feedback_path)
print('run_log:', store.run_log_path)